In [1]:
# Install packages and NLTK resources
!pip install -q pandas openpyxl nltk textblob matplotlib plotly pydub SpeechRecognition unidecode
!apt-get -qq update && apt-get -qq install -y ffmpeg

import nltk
nltk.download('vader_lexicon')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 11.0 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [2]:
# Imports and helper functions
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
from unidecode import unidecode
from IPython.display import display, HTML
from pydub import AudioSegment
import speech_recognition as sr

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Basic labelers
def vader_label(text, threshold_compound=0.05):
    if not isinstance(text, str) or text.strip()=="":
        return "Neutral"
    vs = sia.polarity_scores(text)
    c = vs['compound']
    if c >= threshold_compound:
        return "Positive"
    elif c <= -threshold_compound:
        return "Negative"
    else:
        return "Neutral"

def textblob_label(text):
    if not isinstance(text, str) or text.strip()=="":
        return "Neutral"
    p = TextBlob(text).sentiment.polarity
    if p > 0:
        return "Positive"
    elif p < 0:
        return "Negative"
    else:
        return "Neutral"

def combined_label(text):
    v = vader_label(text)
    if v == "Neutral":
        return textblob_label(text)
    return v

# Cleaning helpers
def clean_text_basic(s):
    if not isinstance(s, str):
        return ""
    s = unidecode(s)
    s = s.replace("\n"," ").replace("\r"," ")
    s = re.sub(r'\s+', ' ', s).strip()
    return s

NEG_KEYWORDS = [
    r'\bbad\b', r'\bpoor\b', r'\bworst\b', r'can do (a )?lot better',
    r'nothing much to like', r'needs improvement', r'\bimprove\b', r'not good',
    r'not helpful', r'\bboring\b', r'\bshould improve\b'
]
POS_KEYWORDS = [
    r'\bgood\b', r'\bexcellent\b', r'\bfabulous\b', r'\bamazing\b', r'\bgreat\b'
]

def rule_override(text, current):
    txt = str(text).lower()
    for pat in NEG_KEYWORDS:
        if re.search(pat, txt):
            return "Negative"
    for pat in POS_KEYWORDS:
        if re.search(pat, txt):
            return "Positive"
    if re.search(r'\bokay\b|\bokish\b', txt):
        return "Neutral"
    return current

def last_token_label(text):
    parts = [p.strip() for p in str(text).split('|') if p.strip()]
    if not parts:
        return None
    last = parts[-1].lower()
    if re.search(r'\bbad\b', last): return "Negative"
    if re.search(r'\bpoor\b', last): return "Negative"
    if re.search(r'\bworst\b', last): return "Negative"
    if re.search(r'\bnone\b', last): return "Negative"
    if re.search(r'\bgood\b', last): return "Positive"
    if re.search(r'\bfabulous\b', last): return "Positive"
    if re.search(r'\bfine\b', last): return "Positive"
    if re.search(r'\bokay|okish|ok\b', last): return "Neutral"
    return None

# Audio conversion helper
def convert_to_wav(input_path, out_path="converted.wav", sr=16000):
    audio = AudioSegment.from_file(input_path)
    audio = audio.set_frame_rate(sr).set_channels(1)
    audio.export(out_path, format="wav")
    return out_path

samples = [
    "The session was very helpful and I learned a lot.",
    "This was the worst lecture I've attended.",
    "It was okay, nothing special."
]
for s in samples:
    print(s, "->", combined_label(s))


The session was very helpful and I learned a lot. -> Positive
This was the worst lecture I've attended. -> Negative
It was okay, nothing special. -> Negative


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [13]:
# Upload a CSV or XLSX file with student feedback
from google.colab import files

print("Upload your feedback file (CSV or XLSX).")
uploaded = files.upload()
fname = next(iter(uploaded))
print("Uploaded:", fname)

# Load file
if fname.lower().endswith('.csv'):
    df = pd.read_csv(fname)
else:
    df = pd.read_excel(fname)

print("Loaded dataframe shape:", df.shape)
display(df.head(6))

Upload your feedback file (CSV or XLSX).


Saving sample_student_feedback_30.csv to sample_student_feedback_30 (1).csv
Uploaded: sample_student_feedback_30 (1).csv
Loaded dataframe shape: (29, 7)


,Q1,Q2,Q3,Q4,Q5,Overall_Feedback,Sentiment
0,The instructor explained the concepts clearly.,The practical examples were useful.,The session was well organized.,I understood the topic better after the session.,The pace of teaching was comfortable.,Very informative and helpful session.,Positive
1,The explanation was easy to understand.,The examples were relevant.,The session was structured well.,The instructor answered questions clearly.,The pace was appropriate.,I had a good learning experience.,Positive
2,The instructor was engaging.,The practical demonstration was excellent.,The content was organized clearly.,The topic became easier to understand.,The teaching pace was comfortable.,A useful and engaging session.,Positive
3,The concepts were explained with good examples.,The exercises were helpful.,The session was well planned.,I gained useful knowledge.,The pace was good.,"Overall, the session was very helpful.",Positive
4,The instructor communicated effectively.,The examples helped clarify the topic.,The session was interactive.,I understood most of the concepts.,The pace was comfortable.,Good session with clear explanations.,Positive
5,The topic was interesting.,The practical examples were clear.,The instructor encouraged questions.,The session improved my understanding.,The pace was suitable.,It was an informative learning session.,Positive


In [15]:
# Combine text columns into full_text
if 'reviewText' in df.columns:
    df['full_text'] = df['reviewText'].astype(str)
else:
    text_cols = [c for c in df.columns if df[c].dtype == object]
    print("Combining these text/object columns:", text_cols)
    if not text_cols:
        raise SystemExit("No text/object columns found. Ensure your file has textual feedback columns.")
    df['full_text'] = df[text_cols].astype(str).agg(' | '.join, axis=1)

# Normalize separators & clean
df['full_text'] = df['full_text'].str.replace(r'\s*\|\s*', ' | ', regex=True).str.strip()
df['full_text_clean'] = df['full_text'].apply(clean_text_basic)
df['full_text_clean_lower'] = df['full_text_clean'].str.lower()

display(df[['full_text_clean']].head(8))


Combining these text/object columns: ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Overall_Feedback', 'Sentiment']


,full_text_clean
0,The instructor explained the concepts clearly....
1,The explanation was easy to understand. | The ...
2,The instructor was engaging. | The practical d...
3,The concepts were explained with good examples...
4,The instructor communicated effectively. | The...
5,The topic was interesting. | The practical exa...
6,The teaching was clear and engaging. | The exa...
7,The instructor explained difficult concepts we...


In [17]:
# Run VADER predictions
df['pred_vader'] = df['full_text_clean'].apply(vader_label)
display(df[['full_text_clean','pred_vader']].head(10))

,full_text_clean,pred_vader
0,The instructor explained the concepts clearly....,Positive
1,The explanation was easy to understand. | The ...,Positive
2,The instructor was engaging. | The practical d...,Positive
3,The concepts were explained with good examples...,Positive
4,The instructor communicated effectively. | The...,Positive
5,The topic was interesting. | The practical exa...,Positive
6,The teaching was clear and engaging. | The exa...,Positive
7,The instructor explained difficult concepts we...,Positive
8,The explanation was difficult to follow. | The...,Negative
9,The instructor moved through the topic too qui...,Negative


In [6]:
# Rule-based override
df['pred_rule'] = df.apply(lambda r: rule_override(r['full_text_clean_lower'], r['pred_vader']), axis=1)

# Show changes introduced by rules
changed = df[df['pred_vader'] != df['pred_rule']]
print("Number of rows changed by rule-based overrides:", len(changed))
display(changed[['full_text_clean','pred_vader','pred_rule']].head(12))


Number of rows changed by rule-based overrides: 2


,full_text_clean,pred_vader,pred_rule
17,The content was informative. | The examples we...,Positive,Negative
20,The explanation was adequate. | More examples ...,Positive,Neutral


In [18]:
# Normalize true labels and compare with pred_vader
df['true_label'] = df.get('Sentiment', df.get('sentiment', df.get('label', pd.Series([""]*len(df))))).astype(str).str.strip().str.capitalize()

if df['true_label'].str.len().sum() == 0:
    print("Warning: 'Sentiment' (true labels) appear empty. Check original file.")

print("True label counts:\n", df['true_label'].value_counts(dropna=False))
print("\nPredicted (VADER) counts:\n", df['pred_vader'].value_counts())

mismatch = df[df['true_label'] != df['pred_vader']]
print("\nTotal mismatches (true vs pred_vader):", len(mismatch), "of", len(df))
display(mismatch[['full_text_clean','true_label','pred_vader']].head(12))


True label counts:
 true_label
Positive    12
Negative     9
Neutral      8
Name: count, dtype: int64

Predicted (VADER) counts:
 pred_vader
Positive    21
Negative     7
Neutral      1
Name: count, dtype: int64

Total mismatches (true vs pred_vader): 9 of 29


,full_text_clean,true_label,pred_vader
13,The instructor needed to explain the topic in ...,Negative,Positive
15,The session was acceptable. | Some examples we...,Neutral,Positive
16,The instructor explained the topic adequately....,Neutral,Positive
17,The content was informative. | The examples we...,Neutral,Positive
18,The teaching was satisfactory. | A few example...,Neutral,Positive
20,The explanation was adequate. | More examples ...,Neutral,Positive
21,The instructor answered most questions. | The ...,Neutral,Positive
22,The session covered the expected topics. | The...,Neutral,Positive
27,The session lacked clarity. | More practical e...,Negative,Positive


In [8]:
# last token override
df['last_token_guess'] = df['full_text_clean_lower'].apply(last_token_label)

# final prediction: prefer last_token_guess (if not None) -> then rule override -> then vader
df['final_pred'] = df.apply(lambda r: r['last_token_guess'] if pd.notna(r['last_token_guess']) else r['pred_rule'], axis=1)

# Show a few examples where last_token_guess influenced result
influenced = df[df['last_token_guess'].notna() & (df['last_token_guess'] != df['pred_vader'])]
print("Rows where last-token override applied (sample):", len(influenced))
display(influenced[['full_text_clean','last_token_guess','pred_vader','pred_rule','final_pred']].head(12))


Rows where last-token override applied (sample): 0


,full_text_clean,last_token_guess,pred_vader,pred_rule,final_pred


In [9]:
# Evaluation summary
total = len(df)
correct = (df['final_pred'] == df['true_label']).sum()
acc = correct/total if total else 0.0
print(f"Accuracy after rules & last-token override: {correct}/{total} = {acc:.3f}")

mismatch2 = df[df['final_pred'] != df['true_label']]
print("Remaining mismatches:", len(mismatch2))
display(mismatch2[['full_text_clean','true_label','pred_vader','pred_rule','last_token_guess','final_pred']].head(20))

print("\nCross-tabulation (true_label vs final_pred):")
display(pd.crosstab(df['true_label'], df['final_pred'], margins=True))


Accuracy after rules & last-token override: 21/29 = 0.724
Remaining mismatches: 8


,full_text_clean,true_label,pred_vader,pred_rule,last_token_guess,final_pred
13,The instructor needed to explain the topic in ...,Negative,Positive,Positive,None,Positive
15,The session was acceptable. | Some examples we...,Neutral,Positive,Positive,None,Positive
16,The instructor explained the topic adequately....,Neutral,Positive,Positive,None,Positive
17,The content was informative. | The examples we...,Neutral,Positive,Negative,None,Negative
18,The teaching was satisfactory. | A few example...,Neutral,Positive,Positive,None,Positive
21,The instructor answered most questions. | The ...,Neutral,Positive,Positive,None,Positive
22,The session covered the expected topics. | The...,Neutral,Positive,Positive,None,Positive
27,The session lacked clarity. | More practical e...,Negative,Positive,Positive,None,Positive



Cross-tabulation (true_label vs final_pred):


final_pred,Negative,Neutral,Positive,All
true_label,,,,
Negative,7,0,2,9
Neutral,1,2,5,8
Positive,0,0,12,12
All,8,2,19,29


In [10]:
# Show sample reviews grouped by predicted sentiment
for s in ['Positive','Negative','Neutral']:
    sample = df[df['final_pred']==s].head(5)
    print(f"\n--- {s} (showing up to 5 samples) ---")
    if sample.empty:
        print("No samples for", s)
    else:
        for i, row in sample.iterrows():
            print(f"{i}: {row['full_text_clean'][:300]}")



--- Positive (showing up to 5 samples) ---
0: The instructor explained the concepts clearly. | The practical examples were useful. | The session was well organized. | I understood the topic better after the session. | The pace of teaching was comfortable. | Very informative and helpful session. | Positive
1: The explanation was easy to understand. | The examples were relevant. | The session was structured well. | The instructor answered questions clearly. | The pace was appropriate. | I had a good learning experience. | Positive
2: The instructor was engaging. | The practical demonstration was excellent. | The content was organized clearly. | The topic became easier to understand. | The teaching pace was comfortable. | A useful and engaging session. | Positive
3: The concepts were explained with good examples. | The exercises were helpful. | The session was well planned. | I gained useful knowledge. | The pace was good. | Overall, the session was very helpful. | Positive
4: The instru

In [19]:
# Browser recorder UI (press button to record)
display(HTML("""
<style>#recBtn{padding:10px 12px; font-size:14px}</style>
<div>
  <button id="recBtn">Start Recording (5s)</button>
  <span id="status" style="margin-left:12px"></span>
</div>
<script>
const btn = document.getElementById('recBtn');
const status = document.getElementById('status');
btn.onclick = async () => {
  try {
    btn.disabled = true;
    status.textContent = "Requesting microphone permission...";
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    status.textContent = "Recording for 5 seconds...";
    const mediaRecorder = new MediaRecorder(stream);
    let chunks = [];
    mediaRecorder.ondataavailable = e => chunks.push(e.data);
    mediaRecorder.onstop = e => {
      const blob = new Blob(chunks, { type: 'audio/webm' });
      const url = URL.createObjectURL(blob);
      const a = document.createElement('a');
      a.style.display = 'none';
      a.href = url;
      a.download = 'recorded.webm';
      document.body.appendChild(a);
      a.click();
      a.remove();
      status.textContent = "Recording done — 'recorded.webm' downloaded. Upload it using the next cell.";
      stream.getTracks().forEach(t => t.stop());
    };
    mediaRecorder.start();
    setTimeout(() => { mediaRecorder.stop(); }, 5000);
  } catch (err) {
    console.error(err);
    status.textContent = "Error: " + err.message;
  } finally {
    btn.disabled = false;
  }
};
</script>
"""))


In [20]:
# Upload recorded audio and transcribe (auto-convert -> WAV)
from google.colab import files

print("Upload the audio file you downloaded (e.g., recorded.webm or recorded.wav or mp3/m4a).")
uploaded_audio = files.upload()
afname = next(iter(uploaded_audio))
print("Uploaded:", afname)

# Convert to clean WAV
try:
    wav_path = convert_to_wav(afname, out_path="audio_converted.wav", sr=16000)
    print("Converted to:", wav_path)
except Exception as e:
    print("Conversion error:", e)
    raise

# Transcribe using SpeechRecognition
r = sr.Recognizer()
try:
    with sr.AudioFile(wav_path) as source:
        audio_data = r.record(source)
    transcribed_text = r.recognize_google(audio_data, language='en-IN')
    print("\nTranscribed text:\n", transcribed_text)
except sr.UnknownValueError:
    transcribed_text = ""
    print("Could not understand audio.")
except sr.RequestError as e:
    transcribed_text = ""
    print("RequestError (speech API):", e)

# Sentiment for audio text
audio_sent_vader = vader_label(transcribed_text)
audio_sent_final = rule_override(transcribed_text.lower(), audio_sent_vader)
print("\nAudio Sentiment (vader -> rules):", audio_sent_final)

# show
transcribed_text, audio_sent_final


Upload the audio file you downloaded (e.g., recorded.webm or recorded.wav or mp3/m4a).


Saving recorded.webm to recorded.webm
Uploaded: recorded.webm
Converted to: audio_converted.wav

Transcribed text:
 the session was difficult

Audio Sentiment (vader -> rules): Negative


('the session was difficult', 'Negative')

In [12]:
# Prepare and download final annotated excel
out_cols = list(df.columns)  # include everything present
final_df = df.copy()

# ensure final_pred exists
if 'final_pred' not in final_df.columns:
    final_df['final_pred'] = final_df.get('pred_rule', final_df.get('pred_vader'))

out_name = "final_sentiment_results.xlsx"
final_df.to_excel(out_name, index=False)
print("Saved:", out_name)

from google.colab import files
files.download(out_name)


Saved: final_sentiment_results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>